In [1]:
import os
print(os.getcwd())

c:\Users\KISIA\AppData\Local\Programs\Microsoft VS Code


In [5]:
import os
os.chdir(r"C:\Users\KISIA\Downloads")
print("현재 위치:", os.getcwd())

현재 위치: C:\Users\KISIA\Downloads


In [7]:
import os
os.chdir(r"C:\Users\KISIA\Downloads")
print("현재 위치:", os.getcwd())

import re
import json
import torch
from transformers import BertTokenizer, BertForTokenClassification

model_dir = "./model_klue"       # 본인 실제 경로로 확인/수정
label_file = "./data/label.txt"  # 본인 실제 경로로 확인/수정

def get_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

label_lst = get_labels(label_file)
id2label = {i: lbl for i, lbl in enumerate(label_lst)}
print("라벨 개수:", len(label_lst))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("klue/bert-base")  # 토크나이저는 원본에서
model = BertForTokenClassification.from_pretrained(model_dir, num_labels=len(label_lst))
model.to(device)
model.eval()

print("모델 로드 완료, device:", device)

현재 위치: C:\Users\KISIA\Downloads
라벨 개수: 30


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5276.09it/s]


모델 로드 완료, device: cuda


In [32]:
from transformers import BertTokenizerFast

# offset_mapping을 쓰려면 Fast 토크나이저가 필요함 (기존 tokenizer는 그대로 둬도 됨)
tokenizer_fast = BertTokenizerFast.from_pretrained("klue/bert-base")


def predict_entities(sentence, max_seq_len=50):
    """
    문장을 서브워드 단위로 토크나이즈하고, 각 서브워드의 예측 태그를 원문 문자 위치(offset)와
    함께 반환. 연속된(문자상 공백 없이 붙어있는) 개체 태그 서브워드만 하나로 묶어서
    (start_char, end_char, tag) 리스트로 반환한다.
    -> 이렇게 하면 '김태호입니다' 중 '김태호'만 정확히 개체로 추출되고, 공백으로 떨어진
       단어끼리는(같은 타입이어도) 절대 하나로 안 묶인다.
    """
    encoding = tokenizer_fast(
        sentence,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_seq_len,
        return_tensors="pt",
    )
    offsets = encoding.pop("offset_mapping")[0].tolist()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = model(**encoding).logits
    preds = torch.argmax(logits, dim=2)[0].cpu().numpy()

    entities = []
    cur_start = cur_end = None
    cur_tag = None

    for (s, e), p in zip(offsets, preds):
        if s == e:  # [CLS], [SEP], [PAD] 같은 특수 토큰 (offset이 (0,0))
            if cur_tag is not None:
                entities.append((cur_start, cur_end, cur_tag))
                cur_tag = None
            continue

        tag = id2label[p]
        if tag in ("O", "UNK"):
            if cur_tag is not None:
                entities.append((cur_start, cur_end, cur_tag))
                cur_tag = None
            continue

        if cur_tag is not None and s == cur_end:
            # 바로 이어지는(공백 없는) 서브워드 -> 같은 브래킷으로 확장
            cur_end = e
        else:
            if cur_tag is not None:
                entities.append((cur_start, cur_end, cur_tag))
            cur_start, cur_end, cur_tag = s, e, tag

    if cur_tag is not None:
        entities.append((cur_start, cur_end, cur_tag))

    return entities  # [(start_char, end_char, tag), ...]


def predict_entities_long(sentence, max_seq_len=50, chunk_words=40):
    """긴 문장은 어절 단위로 청크 분할 후 처리, 각 청크의 char offset을 원문 기준으로 보정."""
    word_spans = [(m.group(), m.start(), m.end()) for m in re.finditer(r'\S+', sentence)]
    if len(word_spans) <= chunk_words:
        return predict_entities(sentence, max_seq_len=max_seq_len)

    all_entities = []
    for i in range(0, len(word_spans), chunk_words):
        chunk = word_spans[i:i + chunk_words]
        chunk_start_char = chunk[0][1]
        chunk_end_char = chunk[-1][2]
        chunk_text = sentence[chunk_start_char:chunk_end_char]  # 원문 그대로 슬라이스 (공백 보존)

        ents = predict_entities(chunk_text, max_seq_len=max_seq_len)
        for s, e, tag in ents:
            all_entities.append((s + chunk_start_char, e + chunk_start_char, tag))

    return all_entities


# 테스트
test_sentence = "안녕하세요, AhnLab의 보안 팀장 김태호입니다. 오늘 아침 8시 45분경, DDoS 공격이 우리의 서버에 발생했습니다."
ents = predict_entities(test_sentence)
for s, e, tag in ents:
    print(f"[{test_sentence[s:e]}:{tag}]")

[AhnLab:ORG-B]
[보안:CVL-B]
[김태호:PER-B]
[오늘:DAT-B]
[아침:TIM-B]
[8시:TIM-B]
[45:TIM-I]
[DDoS:TRM-B]


In [33]:
PHONE_PATTERN = re.compile(
    r'(?<!\d)(?:01[016789][-\s]?\d{3,4}[-\s]?\d{4})(?!\d)'
    r'|(?<!\d)(?:0(?:2|[3-6][1-5])[-\s]?\d{3,4}[-\s]?\d{4})(?!\d)'
)
EMAIL_PATTERN = re.compile(
    r'[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}'
)
CVE_PATTERN = re.compile(r'CVE-\d{4}-\d{4,7}')
_octet = r'(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)'
IPV4_PATTERN = re.compile(rf'(?<![\d.]){_octet}\.{_octet}\.{_octet}\.{_octet}(?![\d.])')

REGEX_DETECTORS = [("PHONE", PHONE_PATTERN), ("EMAIL", EMAIL_PATTERN),
                    ("CVE", CVE_PATTERN), ("IPV4", IPV4_PATTERN)]

def detect_regex(sentence):
    candidates = []
    for label, pattern in REGEX_DETECTORS:
        for m in pattern.finditer(sentence):
            candidates.append((m.start(), m.end(), label))
    candidates.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    resolved, last_end = [], -1
    for s, e, label in candidates:
        if s >= last_end:
            resolved.append((s, e, label))
            last_end = e
    return resolved

In [34]:
def merge_detected(sentence, entities, regex_spans=None):
    """
    entities    : predict_entities()/predict_entities_long()의 결과 [(start,end,tag), ...]
    regex_spans : detect_regex()의 결과 [(start,end,label), ...]
    """
    if regex_spans is None:
        regex_spans = detect_regex(sentence)

    def overlaps_regex(s, e):
        return any(not (e <= rs or s >= re_) for rs, re_, _ in regex_spans)

    events = [(rs, re_, "MASK", None) for rs, re_, _l in regex_spans]
    for s, e, tag in entities:
        if not overlaps_regex(s, e):
            events.append((s, e, "NER", (sentence[s:e], tag)))
    events.sort(key=lambda x: x[0])

    out, cursor = [], 0
    for s, e, kind, payload in events:
        if s < cursor:
            continue
        out.append(sentence[cursor:s])
        out.append("[MASK]" if kind == "MASK" else f"[{payload[0]}:{payload[1]}]")
        cursor = e
    out.append(sentence[cursor:])
    return "".join(out)


# 테스트
print(merge_detected(test_sentence, predict_entities(test_sentence)))

안녕하세요, [AhnLab:ORG-B]의 [보안:CVL-B] 팀장 [김태호:PER-B]입니다. [오늘:DAT-B] [아침:TIM-B] [8시:TIM-B] [45:TIM-I]분경, [DDoS:TRM-B] 공격이 우리의 서버에 발생했습니다.


In [23]:
print(merge_detected(test_sentence, predict_sentence(test_sentence)))

안녕하세요, [AhnLab의:ORG-B] [보안:CVL-B] 팀장 [김태호입니다.:PER-B] [오늘:DAT-B] [아침:TIM-B] [8시:TIM-B] [45분경,:TIM-I] [DDoS:TRM-B] 공격이 우리의 서버에 발생했습니다.


In [36]:
def build_submission(input_path="final_evaluation.json",
                      submission_path="submission.json",
                      label_path="label.txt"):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = []
    used_labels = set()
    label_count = {}
    regex_stats = {"PHONE": 0, "EMAIL": 0, "CVE": 0, "IPV4": 0}
    ner_total = 0

    for item in data:
        idx = item["idx"]
        sentence = item["sentence"]

        entities = predict_entities_long(sentence)
        regex_spans = detect_regex(sentence)

        for _s, _e, label in regex_spans:
            regex_stats[label] = regex_stats.get(label, 0) + 1

        detected = merge_detected(sentence, entities, regex_spans)

        for s, e, tag in entities:
            used_labels.add(tag)
            label_count[tag] = label_count.get(tag, 0) + 1
            ner_total += 1

        results.append({"idx": idx, "sentence": sentence, "detected sentence": detected})

    with open(submission_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    with open(label_path, "w", encoding="utf-8") as f:
        f.write(", ".join(sorted(used_labels)))

    print("=== 실행 통계 (evidence.png 스크린샷용) ===")
    print(f"NER 탐지 개체 전체 수: {ner_total}")
    for tag, cnt in sorted(label_count.items()):
        print(f"  {tag}: {cnt}")
    print(f"정규식 - 연락처:{regex_stats['PHONE']} 이메일:{regex_stats['EMAIL']} "
          f"CVE:{regex_stats['CVE']} IPv4:{regex_stats['IPV4']}")
    print(f"\n완료: {submission_path}, {label_path} 저장됨")
    return results

In [37]:
results = build_submission("final_evaluation.json")

=== 실행 통계 (evidence.png 스크린샷용) ===
NER 탐지 개체 전체 수: 2760
  AFW-B: 27
  AFW-I: 21
  ANM-B: 2
  CVL-B: 214
  CVL-I: 14
  DAT-B: 184
  DAT-I: 244
  EVT-B: 9
  EVT-I: 14
  LOC-B: 283
  LOC-I: 1
  MAT-B: 3
  NUM-B: 218
  NUM-I: 29
  ORG-B: 434
  ORG-I: 33
  PER-B: 199
  PER-I: 9
  PLT-B: 2
  TIM-B: 88
  TIM-I: 89
  TRM-B: 534
  TRM-I: 109
정규식 - 연락처:74 이메일:89 CVE:87 IPv4:111

완료: submission.json, label.txt 저장됨


In [38]:
import os
print(os.listdir("./data"))

['cached_naver-ner_bert-base_50_test', 'cached_naver-ner_bert-base_50_train', 'cached_naver-ner_kobert_128_test', 'cached_naver-ner_kobert_128_train', 'cached_naver-ner_kobert_50_test', 'cached_naver-ner_kobert_50_train', 'cached_naver-ner_roberta-base_50_test', 'cached_naver-ner_roberta-base_50_train', 'label.txt', 'test.tsv', 'train.tsv']


In [39]:
print(sum(1 for line in open("./data/test.tsv", encoding="utf-8") if line.strip()==""))

0


In [40]:
with open("./data/test.tsv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 10:
            break

'나아가 한 스트로크를 하는 제한시간에 대한 지침도 있다 .\tO NUM-B NUM-I O O O O O O\n'
'그건 관두 영업적 득공을 버리고 퇴은을 원하는 엘레나 대행 궐녀를 닮은 여인들을 교련 시켜서 ．\tO O O O O O O PER-B O O O CVL-B O O O\n'
"송인영은 5년 근화향에서 개최된 CJ나인브릿지클래식 토요돌봄으로 'LPGA 직행티킷'을 거머쥐었던 투표권 .\tPER-B DAT-B LOC-B O EVT-B CVL-B ORG-B O O CVL-B O\n"
"역전패' 단국 월드리그 7연패…장영달회장 인퇴\tO EVT-B EVT-I NUM-B O\n"
"‘국내 최대 통신그룹인 KT호의 선장은 누가 될까 . '\tO O O ORG-B CVL-B O O O O\n"
'해자부래기 등을 연출한 무하마드 무설탕껌의 신작인 이 소작은 특유의 희극적한 배역으로 돌아온 제향기르를 비롯해서 막강한 내공을 자랑하는 일원들의 다면으로 관념을 끌고 있습니다 .\tO O O PER-B CVL-B O O O O O O O PER-B O O O O CVL-B O O O O O\n'
'다들 보면, 보지는 못하더라도 안부전화 하고 에베레스트에서 한류스타들이라고, 잘나가는 분들이어서 개개인적으로 감정이 좋고 그렇습니다 .\tO O O O O O LOC-B CVL-B O O O O O O O\n'
'-5명의 해외 익명 부녀회원이 합동 연출, 에로스라는 분수로 7가지의 일담을 담아낸 무비 오감도 .\tNUM-B O O CVL-B O O PER-B O NUM-B O O FLD-B O O\n'
'칼럼의 제호를 ‘믹스트존’이 아니라 ‘철망 밖’이라고 해야겠다 .\tO O O O O O O O\n'
'한편 물음이 있다면 공중이 필수하다는 거예요 .\tNUM-B O O O O O O\n'
'그 중에서도 운이 좋았던 팀, 승운이 따랐던 팀, 실적이 인상할 가능성이 높은 팀은 어디일까 .\tO O O O O O O O O O O O O O O\n'


In [41]:
import os
from seqeval.metrics import precision_score, recall_score, f1_score

test_path = "./data/test.tsv"

sentences, gold_labels = [], []
with open(test_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) != 2:
            continue
        sent_text, tag_text = parts
        words = sent_text.split()
        tags = tag_text.split()
        if len(words) != len(tags):
            continue  # 어절 수와 태그 수 안 맞는 줄은 스킵
        sentences.append(words)
        gold_labels.append(tags)

print(f"테스트 문장 수: {len(sentences)}")

all_preds, all_golds = [], []
for words, gold in zip(sentences, gold_labels):
    sentence = " ".join(words)
    pred_pairs = predict_sentence_long(sentence)
    pred_tags = [t for _, t in pred_pairs]

    min_len = min(len(pred_tags), len(gold))
    all_preds.append(pred_tags[:min_len])
    all_golds.append(gold[:min_len])

precision = precision_score(all_golds, all_preds)
recall = recall_score(all_golds, all_preds)
f1 = f1_score(all_golds, all_preds)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

테스트 문장 수: 9000
Precision: 0.8152
Recall:    0.7674
F1-score:  0.7906


In [42]:
def convert_labels(tag_seqs):
    converted = []
    for tags in tag_seqs:
        new_tags = []
        for t in tags:
            if t == "O" or t == "UNK":
                new_tags.append("O")
            elif t.endswith("-B"):
                new_tags.append("B-" + t[:-2])
            elif t.endswith("-I"):
                new_tags.append("I-" + t[:-2])
            else:
                new_tags.append("O")
        converted.append(new_tags)
    return converted

fixed_golds = convert_labels(all_golds)
fixed_preds = convert_labels(all_preds)

precision = precision_score(fixed_golds, fixed_preds)
recall = recall_score(fixed_golds, fixed_preds)
f1 = f1_score(fixed_golds, fixed_preds)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Precision: 0.8644
Recall:    0.8172
F1-score:  0.8401


In [44]:
print("=" * 50)
print("학생 이름: 형지원")
print("사용 모델: klue/bert-base (checkpoint: ./model_klue)")
print("=" * 50)
print("NER 탐지 개체 전체 수: 2760")
print("  AFW-B: 27, AFW-I: 21, ANM-B: 2, CVL-B: 214, CVL-I: 14,")
print("  DAT-B: 184, DAT-I: 244, EVT-B: 9, EVT-I: 14, LOC-B: 283,")
print("  LOC-I: 1, MAT-B: 3, NUM-B: 218, NUM-I: 29, ORG-B: 434,")
print("  ORG-I: 33, PER-B: 199, PER-I: 9, PLT-B: 2, TIM-B: 88,")
print("  TIM-I: 89, TRM-B: 534, TRM-I: 109")
print("-" * 50)
print("정규식 탐지 - 연락처: 74, 이메일: 89, CVE: 87, IPv4: 111")
print("-" * 50)
print(f"자체 테스트 Precision: {precision:.4f}")
print(f"자체 테스트 Recall:    {recall:.4f}")
print(f"자체 테스트 F1-score:  {f1:.4f}")
print("=" * 50)

학생 이름: 형지원
사용 모델: klue/bert-base (checkpoint: ./model_klue)
NER 탐지 개체 전체 수: 2760
  AFW-B: 27, AFW-I: 21, ANM-B: 2, CVL-B: 214, CVL-I: 14,
  DAT-B: 184, DAT-I: 244, EVT-B: 9, EVT-I: 14, LOC-B: 283,
  LOC-I: 1, MAT-B: 3, NUM-B: 218, NUM-I: 29, ORG-B: 434,
  ORG-I: 33, PER-B: 199, PER-I: 9, PLT-B: 2, TIM-B: 88,
  TIM-I: 89, TRM-B: 534, TRM-I: 109
--------------------------------------------------
정규식 탐지 - 연락처: 74, 이메일: 89, CVE: 87, IPv4: 111
--------------------------------------------------
자체 테스트 Precision: 0.8644
자체 테스트 Recall:    0.8172
자체 테스트 F1-score:  0.8401
